# Svalbard Ground Station Access

Computes all contact windows between a sun-synchronous satellite and the
**Svalbard Satellite Station (SvalSat)** over a 7-day period.

**Scenario**
- 550 km sun-synchronous orbit, LTDN 10:30 (descending node)
- Ground station: SvalSat, 78.229°N / 15.407°E / 500 m altitude
- Minimum elevation: 5°
- Analysis window: 2025-03-01 → 2025-03-08 (7 days)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime

from missiontools import Spacecraft, GroundStation

## 1. Spacecraft

A 550 km SSO with a 10:30 local time at descending node, propagated with J2.

In [ ]:
EPOCH = np.datetime64('2025-03-01T00:00:00', 'us')

sc = Spacecraft.sunsync(
    altitude_km    = 550.0,
    node_solar_time = '10:30',
    node_type      = 'descending',
    epoch          = EPOCH,
)

period_s = 2 * np.pi * np.sqrt(sc.a**3 / sc.central_body_mu)

print(f"Semi-major axis : {sc.a / 1e3:.1f} km")
print(f"Altitude        : {(sc.a - sc.central_body_radius) / 1e3:.1f} km")
print(f"Inclination     : {np.degrees(sc.i):.3f}°")
print(f"RAAN at epoch   : {np.degrees(sc.raan):.3f}°")
print(f"Orbital period  : {period_s / 60:.2f} min")
print(f"Propagator      : {sc.propagator_type}")

## 2. Ground Station

SvalSat sits on the island of Spitsbergen at 78.229°N — so far north that
it can see every polar-orbiting satellite at least twice per day.

In [ ]:
gs = GroundStation(lat=78.229, lon=15.407, alt=500.0)

print(f"Ground station  : SvalSat")
print(f"Latitude        : {gs.lat:.3f}°N")
print(f"Longitude       : {gs.lon:.3f}°E")
print(f"Altitude        : {gs.alt:.0f} m a.s.l.")

## 3. Compute Access Windows

`GroundStation.access()` scans the analysis window at the requested time step
and returns a list of `(AOS, LOS)` pairs as `datetime64[us]` tuples.
A 20 s step is fine for detecting passes whose shortest duration exceeds 5 min.

In [ ]:
T_START = np.datetime64('2025-03-01T00:00:00', 'us')
T_END   = np.datetime64('2025-03-08T00:00:00', 'us')

passes = gs.access(
    sc,
    T_START, T_END,
    el_min = 5.0,                      # degrees
    step   = np.timedelta64(20, 's'),
)

print(f"Total passes found: {len(passes)}")

## 4. Access Report

In [ ]:
print(f"  {'#':>3}  {'AOS (UTC)':^26}  {'LOS (UTC)':^26}  {'Duration':>9}")
print(f"  {'─'*3}  {'─'*26}  {'─'*26}  {'─'*9}")

total_s = 0
for idx, (aos, los) in enumerate(passes, 1):
    dur_s = int((los - aos) / np.timedelta64(1, 's'))
    total_s += dur_s
    print(f"  {idx:>3}  {str(aos):^26}  {str(los):^26}  "
          f"{dur_s // 60:3d}m {dur_s % 60:02d}s")

window_s = int((T_END - T_START) / np.timedelta64(1, 's'))
print()
print(f"  Total passes       : {len(passes)}")
print(f"  Total contact time : {total_s // 3600}h "
      f"{(total_s % 3600) // 60}m {total_s % 60:02d}s")
print(f"  Duty cycle         : {total_s / window_s * 100:.2f}%")
print(f"  Mean pass duration : {total_s / len(passes) / 60:.1f} min")

## 5. Pass Timeline

Each bar represents one contact window.  The bar length is proportional to
contact duration.

In [ ]:
# Convert datetime64 to Python datetime for matplotlib
def to_dt(t64):
    return t64.astype('datetime64[ms]').astype(datetime)

fig, ax = plt.subplots(figsize=(12, 4))

for idx, (aos, los) in enumerate(passes):
    dur_s = int((los - aos) / np.timedelta64(1, 's'))
    ax.barh(
        0,
        left  = to_dt(aos),
        width = to_dt(los) - to_dt(aos),
        height = 0.5,
        color  = 'tab:blue',
        alpha  = 0.7,
    )
    # Annotate short passes with their duration (min:ss)
    mid = to_dt(aos + (los - aos) // 2)
    ax.text(mid, 0, f"{dur_s // 60}m{dur_s % 60:02d}s",
            ha='center', va='center', fontsize=7, color='white', fontweight='bold')

ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
ax.xaxis.set_major_locator(mdates.DayLocator())
ax.set_yticks([])
ax.set_xlim(to_dt(T_START), to_dt(T_END))
ax.set_xlabel('Date (UTC)')
ax.set_title(
    f'SvalSat contact windows — 550 km SSO LTDN 10:30, el ≥ 5°\n'
    f'{len(passes)} passes, {total_s / 3600:.1f} h total contact time'
)
ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Passes per Day

In [ ]:
from collections import defaultdict

passes_per_day = defaultdict(list)
for aos, los in passes:
    day = str(aos)[:10]   # 'YYYY-MM-DD'
    dur_s = int((los - aos) / np.timedelta64(1, 's'))
    passes_per_day[day].append(dur_s)

print(f"  {'Date':>12}  {'Passes':>7}  {'Contact':>12}")
print(f"  {'─'*12}  {'─'*7}  {'─'*12}")
for day in sorted(passes_per_day):
    n = len(passes_per_day[day])
    t = sum(passes_per_day[day])
    print(f"  {day:>12}  {n:>7}  {t // 60:3d}m {t % 60:02d}s")